The classification model has a problem with pointing its attention to the ROI in the accompanying mask.  
If a photo has a lot of trash, the model is basically saying "yes" trash. if not it says other.  
This needs to be fixed with two things.  
- First im creating examples where the model is strongly disagreeing with me
- Im going to assign more weight to those examples

The other useful thing would be a pretrained model which knows how to point. I'll train the sample architecture on the shrinked TACO dataset.   
The model would need a size of photo to work with, we going with 224, easy.  

First though, I need to go through the TACO dset to see if I can extract something useful out of it.  

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.disk import DiskImage, DiskBooleanMask
from mtrain.utils import mkdir
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
from mtrain.smallnet.unet.extract.draw import overlay_mask_on_img
from mtrain.utils import show
from fastai.data.core import DataLoaders, default_device
from collections import defaultdict
from torchvision import tv_tensors
from torchvision.transforms import v2
from PIL import Image
from fastai.vision.all import (
    vision_learner,
    mobilenet_v3_small,
    mobilenet_v3_large,
    accuracy,
    F1Score,
    CrossEntropyLossFlat,
    ProgressCallback,
)

In [ ]:
from mtrain.smallnet.unet.extract.query_taco import *
from pycocotools.coco import COCO

TACO = Path("/Users/hariomnarang/Desktop/personal/TACO/data")
ANN_FILE = TACO / "annotations.json"
coco = COCO(ANN_FILE)

In [ ]:
mult_cls_images = get_images_with_multiple_classes(coco)

In [ ]:
len(mult_cls_images)

In [ ]:
show_image_with_boxes(coco, TACO, mult_cls_images[9])

In [ ]:
img_array = load_image(TACO / mult_cls_images[0]["file_name"])
img_array = cv2.resize(img_array, (130, 130))
plt.imshow(img_array)

In [ ]:
from mtrain.smallnet.unet.extract.draw import overlay_mask_on_img
from mtrain.neg_mask.taco.build_dataset import iter_taco_samples

# grab a few samples and visualise them
samples = []
for s in iter_taco_samples(coco, TACO, size=130):
    samples.append(s)
    if len(samples) == 6 * 4:
        break

# print(f"cat_names: {[s.cat_name for s in samples]}")

fig, axes = plt.subplots(len(samples) // 4, 4, figsize=(20, 20))
axes = axes.flatten()
for i, s in enumerate(samples):
    overlaid = overlay_mask_on_img(s.image, s.mask.astype(bool))
    axes[i].imshow(overlaid)
    axes[i].set_title(s.cat_name, fontsize=8)
    axes[i].set_title(s.cat_name, fontsize=8)
# axes[0, 0].set_ylabel("image")
# axes[1, 0].set_ylabel("mask")
plt.tight_layout()
plt.show()

In [ ]:
from mtrain.neg_mask.taco.build_dataset import build_taco_classification_dataset

In [ ]:
out_dir = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/taco"
)
build_taco_classification_dataset(coco, TACO, out_dir, 220)

In [ ]:
import cv2
from mtrain.disk import DiskImage, DiskBooleanMask

OUT_DIR = out_dir

sample_dirs = [d for d in OUT_DIR.rglob("*/") if (d / "image.jpg").exists()]
print(f"Total samples: {len(sample_dirs)}")
print(f"Categories: {sorted({d.parent.name for d in sample_dirs})}")

chosen = random.sample(sample_dirs, 4 * 6)

fig, axes = plt.subplots(len(samples) // 4, 4, figsize=(20, 20))
axes = axes.flatten()
for i, d in enumerate(chosen):
    img = DiskImage.load(d / "image.jpg")
    mask = DiskBooleanMask.load(d / "mask.png")
    overlaid = overlay_mask_on_img(
        img,
        mask.astype(bool),
    )
    axes[i].imshow(overlaid)
    axes[i].set_title(d.parent.name, fontsize=8)
    axes[i].set_axis_off()
# axes[0, 0].set_ylabel("image")
# axes[1, 0].set_ylabel("mask")
plt.tight_layout()
plt.show()

# fig, axes = plt.subplots(2, len(chosen), figsize=(3 * len(chosen), 6))
# for i, d in enumerate(chosen):
#     img = cv2.cvtColor(cv2.imread(str(d / "image.jpg")), cv2.COLOR_BGR2RGB)
#     mask = cv2.imread(str(d / "mask.png"), cv2.IMREAD_GRAYSCALE)
#     axes[0, i].imshow(img)
#     axes[0, i].set_title(d.parent.name, fontsize=7)
#     axes[0, i].axis("off")
#     axes[1, i].imshow(mask, cmap="gray")
#     axes[1, i].axis("off")
# axes[0, 0].set_ylabel("image")
# axes[1, 0].set_ylabel("mask")
# plt.tight_layout()
# plt.show()

# Train model now

In [ ]:
DS_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/taco"
)

In [ ]:
print([d.name for d in DS_DIR.glob("*")])

In [ ]:
LABELS = [
    "Paper cup",
    "Glass bottle",
    "Aluminium foil",
    "Normal paper",
    "Egg carton",
    "Other plastic bottle",
    "Squeezable tube",
    "Metal bottle cap",
    "Aerosol",
    "Drink can",
    "Wrapping paper",
    "Shoe",
    "Rope & strings",
    "Corrugated carton",
    "Food waste",
    "Single-use carrier bag",
    "Aluminium blister pack",
    "Scrap metal",
    "Foam food container",
    "Garbage bag",
    "Broken glass",
    "Foam cup",
    "Spread tub",
    "Other plastic container",
    "Clear plastic bottle",
    "Plastic film",
    "Other plastic",
    "Disposable plastic cup",
    "Crisp packet",
    "Food Can",
    "Glass jar",
    "Metal lid",
    "Other plastic wrapper",
    "Six pack rings",
    "Plastic bottle cap",
    "Tissues",
    "Plastic lid",
    "Drink carton",
    "Other carton",
    "Disposable food container",
    "Styrofoam piece",
    "Plastic straw",
    "Plastic utensils",
    "Glass cup",
    "Unlabeled litter",
    "Meal carton",
    "Pop tab",
    "Toilet tube",
    "Cigarette",
    "Paper straw",
    "Paper bag",
]

In [ ]:
# only keep directories with image count greater than 2

In [ ]:
AREA_THRES = 5


def _is_valid_dir(direc: Path):
    valid = (
        direc.is_dir()
        and (direc / "image.jpg").exists()
        and (direc / "mask.png").exists()
    )
    return valid


def get_ds_dirs(ds_root, labels):
    res = []
    for label in labels:
        cls_root = ds_root / label
        print(cls_root.name)
        if not cls_root.is_dir():
            continue
        for d in cls_root.glob("*"):
            if _is_valid_dir(d):
                res.append(d)
    random.shuffle(res)
    return res


def _label_func(d: Path):
    return Path(d).parent.name


def _validate_labels_in_dirs(dirs, label_by_idx):
    for d in dirs:
        label = _label_func(d)
        if label not in label_by_idx:
            raise Exception(
                f"Label={label} not found for directory={d}. label_by_index={label_by_idx}"
            )


def get_area(d):
    return np.array(Image.open(d / "mask.png").convert("L")).sum()


total_dirs = get_ds_dirs(DS_DIR, LABELS)
areas_and_dirs = [(get_area(d), d) for d in total_dirs]
dirs = [d for (a, d) in areas_and_dirs if a > AREA_THRES or a == 0]

print(f"total directories scanned: {len(total_dirs)}")
print(f"filtered directories: {len(dirs)}")


labels = [_label_func(d) for d in dirs]
train_dirs, valid_dirs = train_test_split(
    dirs, test_size=0.2, stratify=labels, random_state=42
)

In [ ]:
from fastai.vision.all import imagenet_stats
from mtrain.neg_mask.model.dataset import MaskClassificationDataset

train_ds = MaskClassificationDataset(train_dirs, LABELS, train=True)
valid_ds = MaskClassificationDataset(valid_dirs, LABELS, train=False)

In [ ]:
dls = DataLoaders.from_dsets(train_ds, valid_ds, device=default_device())

In [ ]:
from mtrain.neg_mask.model.learner import load_our_learner

learn = load_our_learner(dls, mobilenet_v3_large, None, LABELS)


In [ ]:
learn.fine_tune(1)

In [ ]:
learn.fit_one_cycle(20)

In [ ]:
MODEL_OUT_DIR = mkdir(Path("../../datasets/models/taco_pretrained_mask_classifier"))

In [ ]:
learn.save((MODEL_OUT_DIR / "mobilenet_v3_large_130x130_iter-20").resolve())

In [ ]:
! ls ../../datasets/models/

In [ ]:
from mtrain.neg_mask.model.show import *

In [ ]:
all_preds, all_targs, decoded, all_losses = get_preds_for_ds(learn, valid_ds, 4)

In [ ]:
show_classification_report(all_preds, all_targs, LABELS)